In [1]:
# Cell 0 | 项目根目录与10点留出实验路径

import json
import os
import sys
from pathlib import Path
from typing import Union


def find_project_root(start_path: Union[str, Path] = ".") -> Path:
    """向上查找同时包含 data 和 notebooks 的项目根目录。"""
    p = Path(start_path).resolve()

    for _ in range(20):
        if (p / "data").exists() and (p / "notebooks").exists():
            return p

        if p.parent == p:
            break

        p = p.parent

    raise FileNotFoundError(
        "未找到项目根目录。请确认当前 notebook 位于：\n"
        r"D:\jupyter\IL_ML_Project\notebooks\特征工程"
    )


ROOT = find_project_root(Path.cwd())

EXPERIMENT_ROOT = (
    ROOT
    / "small_paper_data"
    / "viscosity_10point_holdout"
)

ML_READY_DIR = (
    EXPERIMENT_ROOT
    / "ml_ready"
)

ML_READY_PATH = (
    ML_READY_DIR
    / "ml_ready_A.csv"
)

UPSTREAM_INDEX_PATH = (
    ML_READY_DIR
    / "ml_ready_A_available_indices.npz"
)

UPSTREAM_MANIFEST_PATH = (
    ML_READY_DIR
    / "ml_ready_A_manifest.json"
)

FEATURE_DIR = (
    EXPERIMENT_ROOT
    / "features"
)

GNN_DIR = (
    FEATURE_DIR
    / "mpnn_graph"
)

OUT_DIR = (
    EXPERIMENT_ROOT
    / "outputs"
    / "representation_compare"
)

for folder in [
    GNN_DIR,
    OUT_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

# 防止输出误写入原项目 data/features
experiment_text = str(
    EXPERIMENT_ROOT.resolve()
).lower()

if (
    "small_paper_data" not in experiment_text
    or "viscosity_10point_holdout" not in experiment_text
):
    raise RuntimeError(
        "10点留出实验目录异常，已停止运行，"
        "防止覆盖原项目文件。"
    )

required_input_files = [
    ML_READY_PATH,
    UPSTREAM_INDEX_PATH,
    UPSTREAM_MANIFEST_PATH,
]

missing_input_files = [
    path
    for path in required_input_files
    if not path.exists()
]

if missing_input_files:
    raise FileNotFoundError(
        "缺少上一步生成的输入文件：\n"
        + "\n".join(
            str(path)
            for path in missing_input_files
        )
    )

print("python                 =", sys.executable)
print("cwd                    =", os.getcwd())
print("ROOT                   =", ROOT)
print("EXPERIMENT_ROOT        =", EXPERIMENT_ROOT)
print("ML_READY_PATH          =", ML_READY_PATH)
print("UPSTREAM_INDEX_PATH    =", UPSTREAM_INDEX_PATH)
print("UPSTREAM_MANIFEST_PATH =", UPSTREAM_MANIFEST_PATH)
print("GNN_DIR                =", GNN_DIR)
print("OUT_DIR                =", OUT_DIR)

python                 = D:\conda\envs\rdkit-env\python.exe
cwd                    = D:\jupyter\IL_ML_Project\notebooks\小论文_粘度预测_CO2筛选\01_10点留出验证\03_特征工程
ROOT                   = D:\jupyter\IL_ML_Project
EXPERIMENT_ROOT        = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout
ML_READY_PATH          = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready\ml_ready_A.csv
UPSTREAM_INDEX_PATH    = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready\ml_ready_A_available_indices.npz
UPSTREAM_MANIFEST_PATH = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready\ml_ready_A_manifest.json
GNN_DIR                = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\features\mpnn_graph
OUT_DIR                = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare


In [2]:
# Cell 1 | 读取并核对黏度单性质 ml_ready_A

import json
import numpy as np
import pandas as pd


df = pd.read_csv(
    ML_READY_PATH,
    encoding="utf-8-sig",
)

df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)

# 记录每一行在输入 ml_ready_A.csv 中的位置
df["source_row_idx"] = np.arange(
    len(df),
    dtype=int,
)

required_cols = [
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]

missing_cols = [
    col
    for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"ml_ready_A.csv 缺少必要字段：{missing_cols}"
    )

# 防止误读大论文使用的多性质母表
unrelated_property_cols = [
    col
    for col in [
        "sol",
        "Cp",
        "density",
    ]
    if col in df.columns
]

if unrelated_property_cols:
    raise ValueError(
        f"当前输入仍包含其他性质列：{unrelated_property_cols}。\n"
        "小论文10点留出实验必须读取黏度单性质 ml_ready_A.csv。"
    )

for col in [
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce",
    )

invalid_mask = (
    df["IL_key"].isna()
    | df["cation_smiles"].isna()
    | df["anion_smiles"].isna()
    | df["T_rep"].isna()
    | df["P_rep"].isna()
    | df["eta"].isna()
    | (df["eta"] <= 0)
    | df["lg_eta"].isna()
)

if invalid_mask.any():
    invalid_path = (
        OUT_DIR
        / "invalid_ml_ready_rows.csv"
    )

    invalid_cols = [
        "source_row_idx",
        "IL_key",
        "cation_smiles",
        "anion_smiles",
        "T_rep",
        "P_rep",
        "eta",
        "lg_eta",
    ]

    df.loc[
        invalid_mask,
        invalid_cols,
    ].to_csv(
        invalid_path,
        index=False,
        encoding="utf-8-sig",
    )

    raise ValueError(
        f"发现 {int(invalid_mask.sum())} 行无效输入数据。\n"
        f"明细已保存至：{invalid_path}"
    )

# 核对 lg_eta 是否确实等于 log10(eta)
lg_eta_recalculated = np.log10(
    df["eta"].to_numpy(dtype=float)
)

lg_eta_difference = np.abs(
    df["lg_eta"].to_numpy(dtype=float)
    - lg_eta_recalculated
)

max_lg_eta_difference = float(
    np.nanmax(lg_eta_difference)
)

if max_lg_eta_difference > 1e-8:
    raise ValueError(
        "lg_eta 与 log10(eta) 不一致。\n"
        f"最大绝对差值为：{max_lg_eta_difference}"
    )

if "has_eta" in df.columns:
    has_eta = pd.to_numeric(
        df["has_eta"],
        errors="coerce",
    )

    if not has_eta.fillna(0).eq(1).all():
        raise ValueError(
            "has_eta 中存在非 1 值，"
            "当前输入不是完整的黏度单性质建模表。"
        )

with open(
    UPSTREAM_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file:
    upstream_manifest = json.load(file)

if (
    upstream_manifest.get("target_property")
    != "viscosity"
):
    raise ValueError(
        "上游 manifest 中的 target_property "
        "不是 viscosity。"
    )

if (
    upstream_manifest.get("target_for_modeling")
    != "lg_eta"
):
    raise ValueError(
        "上游 manifest 中的 target_for_modeling "
        "不是 lg_eta。"
    )

if int(
    upstream_manifest.get(
        "n_rows",
        -1,
    )
) != len(df):
    raise ValueError(
        "上游 manifest 中的 n_rows "
        "与当前 ml_ready_A.csv 行数不一致。"
    )

target_cols = {
    "eta": "eta",
    "lg_eta": "lg_eta",
}

print("上游 manifest 核对通过 ✓")
print("df shape              =", df.shape)
print("unique ILs            =", df["IL_key"].nunique())
print("unique cations        =", df["cation_smiles"].nunique())
print("unique anions         =", df["anion_smiles"].nunique())
print("eta range             =", (df["eta"].min(), df["eta"].max()))
print("lg_eta range          =", (df["lg_eta"].min(), df["lg_eta"].max()))
print("max |lg_eta-log10η|   =", max_lg_eta_difference)
print("target columns        =", target_cols)
print("Cell 1 check passed ✓")

上游 manifest 核对通过 ✓
df shape              = (16216, 16)
unique ILs            = 1295
unique cations        = 583
unique anions         = 202
eta range             = (0.1, 2000000000000.0)
lg_eta range          = (-1.0, 12.30102999566398)
max |lg_eta-log10η|   = 1.7763568394002505e-15
target columns        = {'eta': 'eta', 'lg_eta': 'lg_eta'}
Cell 1 check passed ✓


In [3]:
# Cell 2 | 构建并核对黏度可用样本索引

import numpy as np
import pandas as pd


eta_values = pd.to_numeric(
    df["eta"],
    errors="coerce",
)

lg_eta_values = pd.to_numeric(
    df["lg_eta"],
    errors="coerce",
)

eta_mask = (
    eta_values.notna()
    & (eta_values > 0)
).to_numpy()

lg_eta_mask = (
    lg_eta_values.notna()
).to_numpy()

idxs = {
    "eta": np.where(
        eta_mask
    )[0].astype(int),

    "lg_eta": np.where(
        lg_eta_mask
    )[0].astype(int),
}

with np.load(
    UPSTREAM_INDEX_PATH
) as upstream_indices:

    required_keys = [
        "eta_idx",
        "lg_eta_idx",
    ]

    missing_keys = [
        key
        for key in required_keys
        if key not in upstream_indices.files
    ]

    if missing_keys:
        raise KeyError(
            f"上游索引文件缺少：{missing_keys}\n"
            f"文件路径：{UPSTREAM_INDEX_PATH}"
        )

    upstream_eta_idx = (
        upstream_indices["eta_idx"]
        .astype(int)
    )

    upstream_lg_eta_idx = (
        upstream_indices["lg_eta_idx"]
        .astype(int)
    )

if not np.array_equal(
    upstream_eta_idx,
    idxs["eta"],
):
    raise ValueError(
        "当前 eta_idx 与上游索引不一致。"
    )

if not np.array_equal(
    upstream_lg_eta_idx,
    idxs["lg_eta"],
):
    raise ValueError(
        "当前 lg_eta_idx 与上游索引不一致。"
    )

if (
    len(idxs["eta"]) != len(df)
    or len(idxs["lg_eta"]) != len(df)
):
    raise ValueError(
        "黏度单性质 ml_ready_A.csv 中仍有目标值缺失。"
    )

output_index_path = (
    OUT_DIR
    / "ml_ready_A_available_indices.npz"
)

np.savez_compressed(
    output_index_path,
    eta_idx=idxs["eta"],
    lg_eta_idx=idxs["lg_eta"],
)

print("上游黏度索引核对通过 ✓")
print("eta_idx rows    =", len(idxs["eta"]))
print("lg_eta_idx rows =", len(idxs["lg_eta"]))
print("saved:", output_index_path)

上游黏度索引核对通过 ✓
eta_idx rows    = 16216
lg_eta_idx rows = 16216
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare\ml_ready_A_available_indices.npz


In [4]:
# Cell 3 | 清理并解析阳离子和阴离子 SMILES

import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit import RDLogger


RDLogger.DisableLog("rdApp.*")

df["cation_smiles"] = (
    df["cation_smiles"]
    .fillna("")
    .astype(str)
    .str.strip()
)

df["anion_smiles"] = (
    df["anion_smiles"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# 保留原 notebook 中的已知 TFSI 修正规则
TFSI_OK = (
    "O=S(=O)(C(F)(F)F)[N-]"
    "S(=O)(=O)C(F)(F)F"
)

bad_tfsi_variants = {
    "[N-](S(=O)2CF3)2": TFSI_OK,
    "[N-](S(=O)2C(F)(F)F)2": TFSI_OK,
    "[N-](S(=O)2C(F)3)2": TFSI_OK,
    "TFSI": TFSI_OK,
    "Tf2N": TFSI_OK,
    "Tf2N-": TFSI_OK,
}

n_tfsi_fixed = int(
    df["anion_smiles"]
    .isin(bad_tfsi_variants)
    .sum()
)

df["anion_smiles"] = (
    df["anion_smiles"]
    .replace(bad_tfsi_variants)
)

bad_positions = []
mol_c_list = []
mol_a_list = []

for position, (
    cation_smiles,
    anion_smiles,
) in enumerate(
    zip(
        df["cation_smiles"],
        df["anion_smiles"],
    )
):
    mol_c = Chem.MolFromSmiles(
        cation_smiles
    )

    mol_a = Chem.MolFromSmiles(
        anion_smiles
    )

    if (
        mol_c is None
        or mol_a is None
    ):
        bad_positions.append(position)
        mol_c_list.append(None)
        mol_a_list.append(None)

    else:
        mol_c_list.append(mol_c)
        mol_a_list.append(mol_a)

df["mol_c"] = mol_c_list
df["mol_a"] = mol_a_list

bad_report_cols = [
    "source_row_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]

bad_report = (
    df.iloc[bad_positions]
    .loc[:, bad_report_cols]
    .copy()
)

bad_report_path = (
    OUT_DIR
    / "bad_smiles_feature_representation.csv"
)

# 即使没有坏 SMILES，也保存一个仅含表头的文件
bad_report.to_csv(
    bad_report_path,
    index=False,
    encoding="utf-8-sig",
)

df_ok = (
    df[
        df["mol_c"].notna()
        & df["mol_a"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

if len(df_ok) == 0:
    raise RuntimeError(
        "没有可用于构建 MPNN 分子图的有效样本。"
    )

# 下游只需普通字段，不将 RDKit Mol 对象写入 CSV
df_ok_output = (
    df_ok
    .drop(
        columns=[
            "mol_c",
            "mol_a",
        ],
        errors="ignore",
    )
    .copy()
)

df_ok_path = (
    OUT_DIR
    / "df_ok_feature_representation.csv"
)

df_ok_output.to_csv(
    df_ok_path,
    index=False,
    encoding="utf-8-sig",
)

ok_row_idx = (
    df_ok["source_row_idx"]
    .to_numpy(dtype=int)
)

df_ok_row_idx_path = (
    OUT_DIR
    / "df_ok_row_idx.npz"
)

np.savez_compressed(
    df_ok_row_idx_path,
    ok_row_idx=ok_row_idx,
)

print("TFSI variants fixed =", n_tfsi_fixed)
print("bad pairs           =", len(bad_positions))
print("df_ok shape          =", df_ok.shape)
print("saved:", bad_report_path)
print("saved:", df_ok_path)
print("saved:", df_ok_row_idx_path)

TFSI variants fixed = 0
bad pairs           = 0
df_ok shape          = (16216, 18)
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare\bad_smiles_feature_representation.csv
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare\df_ok_feature_representation.csv
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare\df_ok_row_idx.npz


In [5]:
# Cell 4 | 定义 MPNN 分子图构建规则

import torch

from rdkit import Chem
from rdkit.Chem import rdchem
from torch_geometric.data import Data


HYB_MAP = {
    rdchem.HybridizationType.SP: 0,
    rdchem.HybridizationType.SP2: 1,
    rdchem.HybridizationType.SP3: 2,
    rdchem.HybridizationType.SP3D: 3,
    rdchem.HybridizationType.SP3D2: 4,
}

BT_MAP = {
    rdchem.BondType.SINGLE: 0,
    rdchem.BondType.DOUBLE: 1,
    rdchem.BondType.TRIPLE: 2,
    rdchem.BondType.AROMATIC: 3,
}


def initialize_mol(
    mol: Chem.Mol,
) -> Chem.Mol:
    """初始化 RDKit 分子的属性缓存和环信息。"""
    mol_copy = Chem.Mol(mol)

    mol_copy.UpdatePropertyCache(
        strict=False,
    )

    Chem.GetSymmSSSR(
        mol_copy
    )

    return mol_copy


def atom_features(
    atom: rdchem.Atom,
) -> list:
    """保持原 notebook 的 7 维原子特征。"""
    return [
        atom.GetAtomicNum(),
        atom.GetTotalDegree(),
        atom.GetFormalCharge(),
        int(atom.GetIsAromatic()),
        int(atom.IsInRing()),
        atom.GetTotalNumHs(
            includeNeighbors=True
        ),
        HYB_MAP.get(
            atom.GetHybridization(),
            5,
        ),
    ]


def bond_features(
    bond: rdchem.Bond,
) -> list:
    """保持原 notebook 的 4 维化学键特征。"""
    return [
        BT_MAP.get(
            bond.GetBondType(),
            4,
        ),
        int(bond.GetIsConjugated()),
        int(bond.IsInRing()),
        int(bond.GetStereo()),
    ]


def mol_to_pyg(
    mol: Chem.Mol,
) -> Data:
    """将一个 RDKit Mol 转换为 PyG Data。"""
    if mol is None:
        return None

    try:
        mol_copy = initialize_mol(
            mol
        )

    except Exception:
        return None

    atom_feature_list = [
        atom_features(atom)
        for atom in mol_copy.GetAtoms()
    ]

    if len(atom_feature_list) == 0:
        return None

    x = torch.tensor(
        atom_feature_list,
        dtype=torch.long,
    )

    edge_index_list = []
    edge_attr_list = []

    for bond in mol_copy.GetBonds():
        begin_idx = (
            bond.GetBeginAtomIdx()
        )

        end_idx = (
            bond.GetEndAtomIdx()
        )

        current_bond_features = (
            bond_features(bond)
        )

        # 无向键转为两条方向相反的边
        edge_index_list.append(
            [
                begin_idx,
                end_idx,
            ]
        )

        edge_attr_list.append(
            current_bond_features
        )

        edge_index_list.append(
            [
                end_idx,
                begin_idx,
            ]
        )

        edge_attr_list.append(
            current_bond_features
        )

    # 单原子离子可能没有化学键
    if len(edge_index_list) == 0:
        edge_index = torch.zeros(
            (2, 0),
            dtype=torch.long,
        )

        edge_attr = torch.zeros(
            (0, 4),
            dtype=torch.long,
        )

    else:
        edge_index = (
            torch.tensor(
                edge_index_list,
                dtype=torch.long,
            )
            .t()
            .contiguous()
        )

        edge_attr = torch.tensor(
            edge_attr_list,
            dtype=torch.long,
        )

    return Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_attr,
    )


print("MPNN 分子图构建规则定义完成 ✓")
print("node feature dim =", 7)
print("edge feature dim =", 4)

MPNN 分子图构建规则定义完成 ✓
node feature dim = 7
edge feature dim = 4


In [6]:
# Cell 5 | 批量构建并保存阳离子图和阴离子图

import numpy as np
import pandas as pd
import torch


required_graph_cols = [
    "IL_key",
    "mol_c",
    "mol_a",
]

missing_graph_cols = [
    col
    for col in required_graph_cols
    if col not in df_ok.columns
]

if missing_graph_cols:
    raise ValueError(
        f"df_ok 缺少构图所需字段：{missing_graph_cols}"
    )

cation_graphs = []
anion_graphs = []

graph_valid = np.ones(
    len(df_ok),
    dtype=bool,
)

for i, (
    mol_c,
    mol_a,
) in enumerate(
    zip(
        df_ok["mol_c"].tolist(),
        df_ok["mol_a"].tolist(),
    )
):
    graph_c = mol_to_pyg(
        mol_c
    )

    graph_a = mol_to_pyg(
        mol_a
    )

    if (
        graph_c is None
        or graph_a is None
    ):
        graph_valid[i] = False

        cation_graphs.append(
            None
        )

        anion_graphs.append(
            None
        )

        continue

    cation_graphs.append(
        graph_c
    )

    anion_graphs.append(
        graph_a
    )

# 保持原 notebook 的保存方式：
# pt 文件中只保存构图成功的图
keep_idx = np.where(
    graph_valid
)[0].astype(int)

cation_graphs_keep = [
    cation_graphs[i]
    for i in keep_idx
]

anion_graphs_keep = [
    anion_graphs[i]
    for i in keep_idx
]

cation_graphs_path = (
    GNN_DIR
    / "cation_graphs.pt"
)

anion_graphs_path = (
    GNN_DIR
    / "anion_graphs.pt"
)

graph_valid_path = (
    GNN_DIR
    / "graph_valid.npz"
)

graph_manifest_path = (
    GNN_DIR
    / "graph_manifest.csv"
)

torch.save(
    cation_graphs_keep,
    cation_graphs_path,
)

torch.save(
    anion_graphs_keep,
    anion_graphs_path,
)

source_row_idx = (
    df_ok["source_row_idx"]
    .to_numpy(dtype=int)
)

np.savez_compressed(
    graph_valid_path,
    valid=graph_valid.astype(
        np.uint8
    ),
    keep_idx=keep_idx,
    source_row_idx=source_row_idx,
    graph_source_row_idx=(
        source_row_idx[keep_idx]
    ),
)

graph_manifest_cols = [
    "source_row_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
]

graph_manifest_df = (
    df_ok.loc[
        :,
        graph_manifest_cols,
    ]
    .copy()
)

graph_manifest_df[
    "graph_valid"
] = graph_valid.astype(int)

# graph_list_idx 用于明确对应 pt 图列表中的位置
graph_manifest_df[
    "graph_list_idx"
] = -1

graph_manifest_df.loc[
    graph_manifest_df[
        "graph_valid"
    ].eq(1),
    "graph_list_idx",
] = np.arange(
    len(keep_idx),
    dtype=int,
)

graph_manifest_df.to_csv(
    graph_manifest_path,
    index=False,
    encoding="utf-8-sig",
)

print("n_total =", len(df_ok))
print(
    "valid   =",
    int(graph_valid.sum()),
    "/",
    len(graph_valid),
)

print("saved:", cation_graphs_path)
print("saved:", anion_graphs_path)
print("saved:", graph_valid_path)
print("saved:", graph_manifest_path)

n_total = 16216
valid   = 16216 / 16216
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\features\mpnn_graph\cation_graphs.pt
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\features\mpnn_graph\anion_graphs.pt
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\features\mpnn_graph\graph_valid.npz
saved: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\features\mpnn_graph\graph_manifest.csv


In [7]:
# Cell 6 | 保存 MPNN 分子图特征化清单

from datetime import datetime
import hashlib
import json


def calculate_sha256(
    path: Path,
) -> str:
    """计算输入文件 SHA256，便于核对实验数据版本。"""
    sha256 = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        for block in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b"",
        ):
            sha256.update(
                block
            )

    return sha256.hexdigest()


feature_manifest = {
    "created_at": datetime.now().isoformat(
        timespec="seconds"
    ),

    "project_root": str(
        ROOT
    ),

    "experiment_root": str(
        EXPERIMENT_ROOT
    ),

    "task": (
        "viscosity_10point_holdout_"
        "mpnn_graph_construction"
    ),

    "input": {
        "ml_ready_A": str(
            ML_READY_PATH
        ),

        "ml_ready_A_sha256": calculate_sha256(
            ML_READY_PATH
        ),

        "upstream_available_indices": str(
            UPSTREAM_INDEX_PATH
        ),

        "upstream_manifest": str(
            UPSTREAM_MANIFEST_PATH
        ),
    },

    "target_property": "viscosity",

    "target_for_modeling": "lg_eta",

    "target_columns": {
        "eta": "eta",
        "lg_eta": "lg_eta",
    },

    "representation_method": "MPNN",

    "fingerprints_used": False,

    "n_input_rows": int(
        len(df)
    ),

    "n_smiles_valid_rows": int(
        len(df_ok)
    ),

    "n_graph_valid_rows": int(
        graph_valid.sum()
    ),

    "mpnn_graph": {
        "cation_graphs": str(
            GNN_DIR
            / "cation_graphs.pt"
        ),

        "anion_graphs": str(
            GNN_DIR
            / "anion_graphs.pt"
        ),

        "graph_valid": str(
            GNN_DIR
            / "graph_valid.npz"
        ),

        "graph_manifest": str(
            GNN_DIR
            / "graph_manifest.csv"
        ),

        "node_feature_dim": 7,

        "edge_feature_dim": 4,

        "separate_cation_anion_graphs": True,

        "saved_graphs": (
            "Only graph-valid samples are stored in "
            "cation_graphs.pt and anion_graphs.pt."
        ),
    },

    "alignment": {
        "available_indices": str(
            OUT_DIR
            / "ml_ready_A_available_indices.npz"
        ),

        "df_ok_row_idx": str(
            OUT_DIR
            / "df_ok_row_idx.npz"
        ),

        "df_ok": str(
            OUT_DIR
            / "df_ok_feature_representation.csv"
        ),

        "bad_smiles_report": str(
            OUT_DIR
            / "bad_smiles_feature_representation.csv"
        ),
    },
}

feature_manifest_path = (
    OUT_DIR
    / "feature_representation_manifest.json"
)

with open(
    feature_manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        feature_manifest,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(
    "representation_method =",
    "MPNN",
)

print(
    "fingerprints_used     =",
    False,
)

print(
    "saved manifest:",
    feature_manifest_path,
)

representation_method = MPNN
fingerprints_used     = False
saved manifest: D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\outputs\representation_compare\feature_representation_manifest.json


In [8]:
# Cell 7 | 检查全部 MPNN 分子图输出与文件衔接

import json
import numpy as np
import pandas as pd
import torch


expected_files = [
    GNN_DIR
    / "cation_graphs.pt",

    GNN_DIR
    / "anion_graphs.pt",

    GNN_DIR
    / "graph_valid.npz",

    GNN_DIR
    / "graph_manifest.csv",

    OUT_DIR
    / "ml_ready_A_available_indices.npz",

    OUT_DIR
    / "df_ok_row_idx.npz",

    OUT_DIR
    / "df_ok_feature_representation.csv",

    OUT_DIR
    / "bad_smiles_feature_representation.csv",

    OUT_DIR
    / "feature_representation_manifest.json",
]

missing_outputs = [
    path
    for path in expected_files
    if not path.exists()
]

if missing_outputs:
    raise FileNotFoundError(
        "以下输出文件缺失：\n"
        + "\n".join(
            str(path)
            for path in missing_outputs
        )
    )


def load_graph_list(
    path: Path,
):
    """兼容不同 PyTorch 版本读取 PyG Data 列表。"""
    try:
        return torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

    except TypeError:
        return torch.load(
            path,
            map_location="cpu",
        )


cation_graphs_check = load_graph_list(
    GNN_DIR
    / "cation_graphs.pt"
)

anion_graphs_check = load_graph_list(
    GNN_DIR
    / "anion_graphs.pt"
)

if (
    len(cation_graphs_check) == 0
    or len(anion_graphs_check) == 0
):
    raise RuntimeError(
        "阳离子图或阴离子图列表为空。"
    )

df_ok_check = pd.read_csv(
    OUT_DIR
    / "df_ok_feature_representation.csv",
    encoding="utf-8-sig",
)

graph_manifest_check = pd.read_csv(
    GNN_DIR
    / "graph_manifest.csv",
    encoding="utf-8-sig",
)

with np.load(
    GNN_DIR
    / "graph_valid.npz"
) as graph_index_data:

    graph_valid_check = (
        graph_index_data[
            "valid"
        ]
        .astype(bool)
    )

    keep_idx_check = (
        graph_index_data[
            "keep_idx"
        ]
        .astype(int)
    )

    source_row_idx_check = (
        graph_index_data[
            "source_row_idx"
        ]
        .astype(int)
    )

    graph_source_row_idx_check = (
        graph_index_data[
            "graph_source_row_idx"
        ]
        .astype(int)
    )

n_graph_valid = int(
    graph_valid_check.sum()
)

if not (
    len(cation_graphs_check)
    == len(anion_graphs_check)
    == n_graph_valid
    == len(keep_idx_check)
    == len(graph_source_row_idx_check)
):
    raise RuntimeError(
        "阳离子图、阴离子图和图索引数量不一致。"
    )

if len(df_ok_check) != len(df_ok):
    raise RuntimeError(
        "保存的 df_ok_feature_representation.csv "
        "与内存中的 df_ok 行数不一致。"
    )

if len(graph_manifest_check) != len(df_ok):
    raise RuntimeError(
        "graph_manifest.csv 行数与 df_ok 不一致。"
    )

required_manifest_cols = [
    "source_row_idx",
    "IL_key",
    "cation_smiles",
    "anion_smiles",
    "T_rep",
    "P_rep",
    "eta",
    "lg_eta",
    "graph_valid",
    "graph_list_idx",
]

missing_manifest_cols = [
    col
    for col in required_manifest_cols
    if col not in graph_manifest_check.columns
]

if missing_manifest_cols:
    raise RuntimeError(
        "graph_manifest.csv 缺少字段："
        f"{missing_manifest_cols}"
    )

manifest_valid_count = int(
    graph_manifest_check[
        "graph_valid"
    ].sum()
)

if (
    manifest_valid_count
    != len(cation_graphs_check)
):
    raise RuntimeError(
        "graph_manifest.csv 中的有效图数量 "
        "与 pt 图列表数量不一致。"
    )

valid_manifest = (
    graph_manifest_check[
        graph_manifest_check[
            "graph_valid"
        ].eq(1)
    ]
    .copy()
)

expected_graph_list_idx = np.arange(
    len(valid_manifest),
    dtype=int,
)

actual_graph_list_idx = (
    valid_manifest[
        "graph_list_idx"
    ]
    .to_numpy(dtype=int)
)

if not np.array_equal(
    actual_graph_list_idx,
    expected_graph_list_idx,
):
    raise RuntimeError(
        "graph_list_idx 与 pt 图列表顺序不一致。"
    )

manifest_graph_source_idx = (
    valid_manifest[
        "source_row_idx"
    ]
    .to_numpy(dtype=int)
)

if not np.array_equal(
    manifest_graph_source_idx,
    graph_source_row_idx_check,
):
    raise RuntimeError(
        "graph_manifest.csv 中的 source_row_idx "
        "与 graph_valid.npz 不一致。"
    )

with np.load(
    OUT_DIR
    / "df_ok_row_idx.npz"
) as df_ok_index_data:

    saved_ok_row_idx = (
        df_ok_index_data[
            "ok_row_idx"
        ]
        .astype(int)
    )

if not np.array_equal(
    saved_ok_row_idx,
    source_row_idx_check,
):
    raise RuntimeError(
        "df_ok_row_idx.npz 与 graph_valid.npz "
        "中的原始行号不一致。"
    )

with np.load(
    OUT_DIR
    / "ml_ready_A_available_indices.npz"
) as target_index_data:

    saved_eta_idx = (
        target_index_data[
            "eta_idx"
        ]
        .astype(int)
    )

    saved_lg_eta_idx = (
        target_index_data[
            "lg_eta_idx"
        ]
        .astype(int)
    )

if not np.array_equal(
    saved_eta_idx,
    idxs["eta"],
):
    raise RuntimeError(
        "保存的 eta_idx 与当前数据不一致。"
    )

if not np.array_equal(
    saved_lg_eta_idx,
    idxs["lg_eta"],
):
    raise RuntimeError(
        "保存的 lg_eta_idx 与当前数据不一致。"
    )

# 检查每一个阳离子图和阴离子图
for graph_name, graph_list in [
    (
        "cation",
        cation_graphs_check,
    ),
    (
        "anion",
        anion_graphs_check,
    ),
]:
    for graph_idx, graph in enumerate(
        graph_list
    ):
        if graph is None:
            raise RuntimeError(
                f"{graph_name} 图列表中存在 None："
                f"graph_idx={graph_idx}"
            )

        if (
            graph.x.ndim != 2
            or graph.x.shape[1] != 7
        ):
            raise RuntimeError(
                f"{graph_name} 图 {graph_idx} "
                "的原子特征维度不是 7。"
            )

        if (
            graph.edge_index.ndim != 2
            or graph.edge_index.shape[0] != 2
        ):
            raise RuntimeError(
                f"{graph_name} 图 {graph_idx} "
                "的 edge_index 形状不正确。"
            )

        if (
            graph.edge_attr.ndim != 2
            or graph.edge_attr.shape[1] != 4
        ):
            raise RuntimeError(
                f"{graph_name} 图 {graph_idx} "
                "的化学键特征维度不是 4。"
            )

        if (
            graph.edge_index.shape[1]
            != graph.edge_attr.shape[0]
        ):
            raise RuntimeError(
                f"{graph_name} 图 {graph_idx} "
                "的边数量与边特征数量不一致。"
            )

with open(
    OUT_DIR
    / "feature_representation_manifest.json",
    "r",
    encoding="utf-8",
) as file:
    manifest_check = json.load(file)

if (
    manifest_check.get("target_property")
    != "viscosity"
):
    raise RuntimeError(
        "manifest 中的 target_property "
        "不是 viscosity。"
    )

if (
    manifest_check.get("target_for_modeling")
    != "lg_eta"
):
    raise RuntimeError(
        "manifest 中的 target_for_modeling "
        "不是 lg_eta。"
    )

if (
    manifest_check.get("representation_method")
    != "MPNN"
):
    raise RuntimeError(
        "manifest 中的 representation_method "
        "不是 MPNN。"
    )

if (
    manifest_check.get("fingerprints_used")
    is not False
):
    raise RuntimeError(
        "manifest 错误地标记为使用了分子指纹。"
    )

if int(
    manifest_check.get(
        "n_input_rows",
        -1,
    )
) != len(df):
    raise RuntimeError(
        "manifest 中的 n_input_rows 不正确。"
    )

if int(
    manifest_check.get(
        "n_smiles_valid_rows",
        -1,
    )
) != len(df_ok):
    raise RuntimeError(
        "manifest 中的 n_smiles_valid_rows 不正确。"
    )

if int(
    manifest_check.get(
        "n_graph_valid_rows",
        -1,
    )
) != len(cation_graphs_check):
    raise RuntimeError(
        "manifest 中的 n_graph_valid_rows 不正确。"
    )

print("=" * 80)
print("全部 MPNN 分子图输出检查通过 ✓")
print("=" * 80)

print("\n输入数据：")
print("input file          =", ML_READY_PATH)
print("input rows          =", len(df))
print("unique ILs          =", df["IL_key"].nunique())
print("unique cations      =", df["cation_smiles"].nunique())
print("unique anions       =", df["anion_smiles"].nunique())

print("\nSMILES 与分子图：")
print("SMILES invalid rows =", len(df) - len(df_ok))
print("SMILES valid rows   =", len(df_ok))
print("graph invalid rows  =", len(df_ok) - n_graph_valid)
print("graph valid rows    =", n_graph_valid)
print("cation graph count  =", len(cation_graphs_check))
print("anion graph count   =", len(anion_graphs_check))
print("node feature dim    =", 7)
print("edge feature dim    =", 4)

print("\n黏度目标索引：")
print("eta_idx rows        =", len(saved_eta_idx))
print("lg_eta_idx rows     =", len(saved_lg_eta_idx))

print("\n表征设置：")
print("representation      =", "MPNN")
print("fingerprints used   =", False)

print("\n输出文件完整路径：")
for path in expected_files:
    print(path)

全部 MPNN 分子图输出检查通过 ✓

输入数据：
input file          = D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\ml_ready\ml_ready_A.csv
input rows          = 16216
unique ILs          = 1295
unique cations      = 583
unique anions       = 202

SMILES 与分子图：
SMILES invalid rows = 0
SMILES valid rows   = 16216
graph invalid rows  = 0
graph valid rows    = 16216
cation graph count  = 16216
anion graph count   = 16216
node feature dim    = 7
edge feature dim    = 4

黏度目标索引：
eta_idx rows        = 16216
lg_eta_idx rows     = 16216

表征设置：
representation      = MPNN
fingerprints used   = False

输出文件完整路径：
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\features\mpnn_graph\cation_graphs.pt
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\features\mpnn_graph\anion_graphs.pt
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\features\mpnn_graph\graph_valid.npz
D:\jupyter\IL_ML_Project\small_paper_data\viscosity_10point_holdout\features\mp